In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import database 
import gameSearch 
import datetime

from database import is_subscribed 
from database import subscriptions 
from database import sub_type 
from database import rental_lim

from datetime import datetime, timedelta

from gameSearch import search_Vid, search_Board


rent_status = None

# Allows user to rent game (if avaliable) and adds it to Rental.txt

def game_Rent(user_ID, game_ID):
    
    global rent_status

    if is_subscribed(user_ID):

        if search_Vid(game_ID) or search_Board(game_ID):

            rent_db = open('Rental.txt', 'r')
            rental_Date = datetime.now().strftime("%Y-%m-%d")

            rent_lim = rental_lim(user_ID)
            year = int(rental_Date.replace('-', ' ').split()[0])
            month = int(rental_Date.replace('-', ' ').split()[1])
            day = int(rental_Date.replace('-', ' ').split()[2])

            latest_return_Date = datetime(year, month, day) + timedelta(weeks=rent_lim)
            latest_return_Date = str(latest_return_Date).replace(" 00:00:00", "")

            for line in rent_db.readlines()[1:]:
                id = line.split(',')[0]
                returned = line.split(',')[2]
                if game_ID == id and returned == 'N/A':
                    rent_status = "already rented"
                    return
        
            else: 
                rent_db.close()
                write_db = open('Rental.txt', 'a')
                entry = f"{game_ID},{rental_Date},N/A,{user_ID},{latest_return_Date}"
                write_db.write(entry)
                write_db.write("\n")
                rent_status = "success"
                return
            
        else: 
            rent_status = "not exist"
            return

    else: 
       rent_status = "not subscribed"
       return
    


# Allows store manager to see what Video games are avaliable for rent

def avaliable_vid_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Video_Game_Info.txt", "r")
    avaliable.append('GameID, Name, Platform, Genre')

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
    
    return avaliable



# Allows store manager to see what Board games are avaliable for rent 

def avaliable_brd_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Board_Game_Info.txt", "r")
    avaliable.append("GameID, Name, No.Players, Genre")

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
      
    return avaliable


In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import datetime 
from datetime import datetime


return_status = None


def game_Return(game_ID, user_ID):

    global return_status

    r = open("Rental.txt", "r")
    file_lst = r.readlines()[1:]
    r.close()


    game_return_history = []
    for line in file_lst:
        if line.split(",")[0] == game_ID:
            game_return_history.append(line.split(",")[2])

    
    if "N/A" in game_return_history:

        w = open("Rental.txt", "w")

        w.write('GameID,RentalDate,ReturnDate,RentalCustomerID,LatestReturnDate\n')

        for line in file_lst: 

            game_id = line.split(",")[0]
            rental_date = line.split(",")[1]
            return_date = line.split(",")[2]
            user_id = line.split(",")[3]
            latest_date = line.split(",")[4]
            current_date = datetime.now().strftime("%Y-%m-%d")

            concat_latest_date = int(latest_date.replace("-", ""))
            concat_return_date = int(current_date.replace("-", ""))

            if all([game_id == game_ID, user_id == user_ID, return_date == "N/A"]):

                w.write(f"{game_id},{rental_date},{current_date},{user_id},{latest_date}")

                if concat_return_date > concat_latest_date: 
                    return_status = "Late"

                elif concat_return_date < concat_latest_date: 
                    return_status = "On Time"            

            else: 
                w.write(line)
    
        w.close()


    else: 
        return_status = "Invalid"


In [ ]:
# Cell defining all the widgets that appear in the GUI (There will be more to come)


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


rent_button = widgets.Button(
    value=False,
    description='Rent Games',
    disabled=False,
)

return_button = widgets.Button(
    value=False,
    description='Return Games',
    disabled=False
)

sessions_button = widgets.Button(
    value=False,
    description='Book Sessions',
    disabled=False,
)

pruning_button = widgets.Button(
    value=False,
    description='Prune Inventory',
    disabled=False
)

exit_button = widgets.Button(
    value=False,
    description='Exit',
    disabled=False
)

game_ID_text = widgets.Text(
    placeholder="Enter Game ID",
    disabled=False
)

user_ID_text = widgets.Text(
    placeholder="Enter User ID",
    disabled=False
)

game_search_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)

view_games_button = widgets.Button(
    value=False, 
    description='Avaliable Games',
    disabled=False
)

board_games_button = widgets.Button(
    value=False, 
    description='Board Games',
    disabled=False
)

video_games_button = widgets.Button(
    value=False, 
    description='Video Games',
    disabled=False
)

game_return_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)

enter_review_text = widgets.Text(
    disabled=False
)

enter_rating_dropdown = widgets.Dropdown(
    options=[('1 Star', 1), ('2 Star', 2), ('3 Star', 3), ('4 Star', 4), ('5 Star', 5)],
    value=1,
    layout=Layout(width="150px"),
    disabled=False
)

yes_review = widgets.Button(
    value=False, 
    description='Yes',
    disabled=False
)

no_review = widgets.Button(
    value=False, 
    description='no',
    disabled=False
)

review_submit = widgets.Button(
    value=False, 
    description="Submit",
    disabled=False
)

on_time = widgets.HTML(value="Game Returned on Time!")

leave_review = widgets.HTML(value="Would you like to leave a review?")

already_rented = widgets.HTML(value="Failed to Rent! Game is already Rented!")

successful_rent = widgets.HTML(value="Game successfully Rented!")

game_not_exist = widgets.HTML(value="Failed to Rent! This game does not exist!")

not_subscribed = widgets.HTML(value="Failed to Rent! User is not Subscribed!")

returned_late = widgets.HTML(value="Game was returned Late!")

not_to_return = widgets.HTML(value="Game is not out for Return!")

comment_and_rate = widgets.HTML(value="Leave a comment & rate the game:")

In [ ]:
# Cell defining the output widgets for each of the widgets:


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox

rent_output = widgets.Output()
return_output = widgets.Output()
pruning_output = widgets.Output()
sessions_output = widgets.Output()
avaliable_games_output = widgets.Output()
board_games_output = widgets.Output()

In [ ]:
# Cell defining the primary boxes of the GUI


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


# Box containing the 5 initial buttons 

button_box = Box(children=[view_games_button, rent_button, return_button, sessions_button, pruning_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width = '200px'))

# Box containing the output widgets for each initial button

output_box = Box(children=[avaliable_games_output, rent_output, return_output, sessions_output, pruning_output],
                 layout=Layout(display='flex',
                               flex_flow='column wrap',
                               align_items='center',
                               align_content='center'))

# Box containing the output widgets for the initial Rent button

rent_box = Box(children=[game_ID_text, user_ID_text, game_search_submit_button, exit_button],
               layout=Layout(display='flex',
                             flex_flow='row wrap',
                             justify_content = 'center',
                             align_items='center',
                             align_content='center',
                             width='200px'))

# Box containing the output widgets for the initial Return button 

return_box = Box(children=[game_ID_text, user_ID_text, game_return_submit_button, review_submit, exit_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='200px'))

# Box containing the output widgets for the initial View Games Button 

view_games_box = Box(children=[board_games_button, video_games_button, exit_button],
                     layout=Layout(display='flex',
                                   flex_flow='row wrap',
                                   justify_content='center',
                                   align_items='center',
                                   align_content='center',
                                   width='200px'))

# Box containing widgets to review games upon return 

review_box = Box(children=[comment_and_rate, enter_review_text, enter_rating_dropdown, exit_button],
                 layout=Layout(display='flex', 
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='300px'))

# Box which contains the entire GUI 

universe_box = Box(children=[button_box, output_box],
                   layout=Layout(display='flex',
                                 flex_flow='row wrap',
                                justify_content='center',
                                align_items='center',
                                align_content='center',
                                width='100%',
                                height='100%'))

In [ ]:
# Cell defining the functions defining the functionality of the widgets 


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


def on_rent_button_clicked(b):

    rent_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    rent_output.layout.display = 'flex'
    rent_output.layout.justify_content='center'
    rent_output.layout.align_items='center'
    rent_output.layout.align_content='center'
    with rent_output:
        display(rent_box)


rent_button.on_click(on_rent_button_clicked)


def on_return_button_clicked(b):

    return_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    return_output.layout.display = 'flex'
    return_output.layout.justify_content='center'
    return_output.layout.align_items='center'
    return_output.layout.align_content='center'
    with return_output:
        display(return_box)


return_button.on_click(on_return_button_clicked)


def on_pruning_button_clicked(b):

    button_box.layout.display = 'none'
    pruning_output.layout.display = 'flex'
    with pruning_output:
        print("This is pruning output")
        display(exit_button)


pruning_button.on_click(on_pruning_button_clicked)


def on_sessions_button_clicked(b):

    button_box.layout.display = 'none'
    sessions_output.layout.display = 'flex'
    with sessions_output:
        print("This is sessions output")
        display(exit_button)


sessions_button.on_click(on_sessions_button_clicked)


def on_avaliable_games_clicked(b):

    video_games_button.layout.display = 'block'
    board_games_button.layout.display = 'block'
    button_box.layout.display = 'none'
    avaliable_games_output.layout.display = 'flex'
    with avaliable_games_output: 
        display(view_games_box) 


view_games_button.on_click(on_avaliable_games_clicked)


def on_board_games_button_clicked(b):

    video_games_button.layout.display = 'none'
    board_games_button.layout.display = 'none'
    with avaliable_games_output: 
        avaliable_board_games = VBox(children=[widgets.Label(entry) for entry in avaliable_brd_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
       
        display(avaliable_board_games)


board_games_button.on_click(on_board_games_button_clicked)


def on_video_games_button_clicked(b):

    video_games_button.layout.display = 'none'
    board_games_button.layout.display = 'none'
    with avaliable_games_output: 
        avaliable_video_games = VBox(children=[widgets.Label(entry) for entry in avaliable_vid_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='300px'))
        
        display(avaliable_video_games)


video_games_button.on_click(on_video_games_button_clicked)


def on_game_search_submit_button_clicked(b):

    game_ID = game_ID_text.value 
    user_ID = user_ID_text.value
    rent_output.clear_output()

    with rent_output: 

        game_Rent(user_ID, game_ID)

        if rent_status == "already rented":
            display(already_rented)
            display(exit_button)
        elif rent_status == "success":
            display(successful_rent)
            display(exit_button) 
        elif rent_status == "not exist": 
            display(game_not_exist) 
            display(exit_button) 
        elif rent_status == "not subscribed":
            display(not_subscribed)
            display(exit_button)


game_search_submit_button.on_click(on_game_search_submit_button_clicked)


def on_game_return_submit_button_clicked(b):

    game_ID = game_ID_text.value 
    user_ID = user_ID_text.value 
    return_output.clear_output()

    with return_output:

        game_Return(game_ID, user_ID)

        if return_status == "On Time":
            display(widgets.VBox(children=[on_time, leave_review, yes_review, no_review],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
 

        elif return_status == "Late": 
            display(returned_late)
            display(exit_button)

        elif return_status == "Invalid":
            display(not_to_return)
            display(exit_button)


game_return_submit_button.on_click(on_game_return_submit_button_clicked)


def on_yes_review_clicked(b):
    
    return_output.clear_output() 

    with return_output: 
        display(review_box)

yes_review.on_click(on_yes_review_clicked)


def on_no_review_clicked(b):

    return_output.clear_output()
    return_output.layout.display='none'
    button_box.layout.display='flex'
    game_ID_text.value = ""
    user_ID_text.value = ""

no_review.on_click(on_no_review_clicked)

def on_review_submit_clicked(b): 
    pass

review_submit.on_click(on_review_submit_clicked)


def on_exit_button_clicked(b):

    sessions_output.clear_output()
    pruning_output.clear_output()
    return_output.clear_output()
    rent_output.clear_output()
    avaliable_games_output.clear_output()

    game_ID_text.value = ""
    user_ID_text.value = ""

    sessions_output.layout.display = 'none'
    pruning_output.layout.display = 'none'
    return_output.layout.display = 'none'
    rent_output.layout.display = 'none'
    avaliable_games_output.display = 'none'
    button_box.layout.display = 'flex'


exit_button.on_click(on_exit_button_clicked)


# Initial state of the GUI

display(universe_box)
